# G1 / G2 — where does the 3.41 ft gap come from, and is the leak available?

Two cheap checks, seconds of CPU each. **G1 appends to the v22 OOF fork** (it
reads `v22_oof.pkl`). **G2 is standalone** — competition data only.

## Why this replaces the test-vs-train comparison I proposed

That plan was to compare test wells' field support against training wells'. It
can't be done: the visible `test/` folder holds 3 wells, and the competition
states those are training instances. Real test coordinates only exist during a
scored rerun. There is no way to see the test spatial distribution from here.

So G1 attacks it from the training side instead, which turns out to be the
stronger test anyway — it produces an **upper bound** rather than a point
estimate.

## G1 — how much of the gap can field support explain, *at most*?

`nn_dist` is the distance from a well to its nearest neighbour in the spatial
map (`predict_spatial_dip` returns it; it drives `spatial_conf`). Your quartile
table already showed error climbing from ~5.3 to 6.49 in the top quartile.

G1 asks the counterfactual directly: **if every test well were as isolated as the
most isolated q% of training wells, what error would the training relationship
predict?** Sweeping q to its extreme gives the largest error increase that
isolation alone can produce.

If that ceiling is well under 3.41 ft, field-support shift is a minority
contributor and the remaining gap is something else — which redirects the search
rather than confirming a hunch. That's worth more than a point estimate.

The bound is generous by construction: it assumes the *entire* test set is as
isolated as your worst training wells, which is far more extreme than any
plausible split.

## G2 — would the public notebooks' contact override fire for you?

Their 6.4x rests on checking whether a test well's 8-character hash also appears
in `train/`, then copying the labels across. G2 runs the same check on your data:
it reads the test well IDs out of `sample_submission.csv` and intersects them
with your training well names.

Three outcomes:

- **Zero overlap, ~200 distinct test IDs** — the exploit is unavailable to you
  interactively, and the public scores likely depend on the rerun swapping in
  wells that do overlap.
- **Substantial overlap** — it is available, and you need to decide whether to
  use it. Note the private-LB fragility: if the private split doesn't share wells
  the way the public one does, anything built on it collapses.
- **Only ~3 test IDs listed** — `sample_submission.csv` covers just the visible
  examples, and the question stays open until a scored run.

Either way this is a read-only probe. It writes nothing.

In [ ]:
# ===== G1: how much of the local->LB gap can field support explain, at most? =====
# Run in the v22 OOF fork (needs v22_oof.pkl). Seconds.
import pickle, glob, os
import numpy as np

LB_SCORE = 8.913          # your best leaderboard score
_p = ([p for p in glob.glob('v22_oof.pkl')] +
      [p for p in glob.glob('/kaggle/input/*/v22_oof.pkl')])
D = pickle.load(open(_p[0], 'rb'))
W = sorted(D['err'])
e  = np.array([D['err'][w] for w in W], float)
nn = np.array([D['nn'][w]  for w in W], float)
fc = np.array([D['field_conf'][w] for w in W], float)

ok = np.isfinite(e) & np.isfinite(nn) & (nn >= 0)
e, nn, fc = e[ok], nn[ok], fc[ok]
local = e.mean(); gap = LB_SCORE - local
print('local %.3f | LB %.3f | gap %.3f ft over %d wells' % (local, LB_SCORE, gap, len(e)))

# --- error as a function of isolation, by decile (robust, non-parametric) -----
print('\n%-24s %7s %9s' % ('nn_dist decile', 'n', 'mean err'))
print('-' * 42)
q = np.quantile(nn, np.linspace(0, 1, 11))
for i in range(10):
    s = (nn >= q[i]) & (nn <= q[i + 1])
    print('  %7.0f - %7.0f ft %7d %9.2f' % (q[i], q[i + 1], s.sum(), e[s].mean()))

# --- counterfactual: all test wells at least as isolated as the q-th pct ------
print('\n%-46s %9s %9s %9s'
      % ('counterfactual: every test well isolated >=', 'E[err]', 'rise', '% of gap'))
print('-' * 78)
rows = []
for pct in [0, 25, 50, 75, 90, 95, 99]:
    thr = np.quantile(nn, pct / 100.0)
    s = nn >= thr
    if s.sum() < 5:
        continue
    m = e[s].mean(); rise = m - local
    rows.append((pct, m, rise))
    print('  %3d-th pct of training wells (%6.0f ft, n=%3d)      %9.3f %9.3f %8.1f%%'
          % (pct, thr, s.sum(), m, rise, 100 * rise / gap))

_, _, max_rise = max(rows, key=lambda r: r[2])
print('\n' + '=' * 78)
print('UPPER BOUND: isolation can explain at most %.3f ft of the %.3f ft gap (%.0f%%).'
      % (max_rise, gap, 100 * max_rise / gap))
print('  This assumes the ENTIRE test set is as isolated as your most isolated')
print('  training wells — far more extreme than any realistic split.')
print('-' * 78)
if max_rise < 0.35 * gap:
    print('VERDICT: field support is a MINOR contributor. At least %.2f ft of the'
          % (gap - max_rise))
    print('  gap is something else — target that instead of the confidence taper.')
elif max_rise < 0.7 * gap:
    print('VERDICT: field support is a PARTIAL contributor. Worth retuning the')
    print('  low-support taper, but it cannot close the gap on its own.')
else:
    print('VERDICT: field support could account for most of the gap. Retuning the')
    print('  confidence taper for isolated wells is the highest-value next move.')
print('=' * 78)

# --- is field_conf a second, separable axis? ---------------------------------
from scipy.stats import spearmanr
print('\nspearman(nn_dist, err)    = %+.3f' % spearmanr(nn, e).statistic)
print('spearman(field_conf, err) = %+.3f' % spearmanr(fc, e).statistic)
print('spearman(nn_dist, field_conf) = %+.3f  (if weak, they are separate axes)'
      % spearmanr(nn, fc).statistic)


In [ ]:
# ===== G2: would the same-well contact override fire on your data? =====
# Standalone — competition data only. Read-only. Seconds.
import glob, os
import pandas as pd

DATA = None
for c in ['/kaggle/input/competitions/rogii-wellbore-geology-prediction',
          '/kaggle/input/rogii-wellbore-geology-prediction']:
    if os.path.isdir(os.path.join(c, 'train')):
        DATA = c; break
if DATA is None:
    _h = glob.glob('/kaggle/input/*/train/*__horizontal_well.csv')
    DATA = os.path.dirname(os.path.dirname(_h[0])) if _h else None
assert DATA, 'competition data not attached'
print('data:', DATA)

train_ids = set(os.path.basename(f).split('__')[0]
                for f in glob.glob(os.path.join(DATA, 'train', '*__horizontal_well.csv')))
test_files = set(os.path.basename(f).split('__')[0]
                 for f in glob.glob(os.path.join(DATA, 'test', '*__horizontal_well.csv')))
print('train wells on disk : %d' % len(train_ids))
print('test  wells on disk : %d   %s'
      % (len(test_files), sorted(test_files) if len(test_files) < 8 else ''))

sub = pd.read_csv(os.path.join(DATA, 'sample_submission.csv'))
sub_ids = sorted(set(sub['id'].astype(str).str[:8]))
print('\nsample_submission.csv : %d rows, %d distinct well ids' % (len(sub), len(sub_ids)))

overlap = sorted(set(sub_ids) & train_ids)
print('\n' + '=' * 72)
print('test ids that ALSO appear in train/ : %d of %d  (%.1f%%)'
      % (len(overlap), len(sub_ids), 100 * len(overlap) / max(len(sub_ids), 1)))
if overlap[:10]:
    print('  e.g. %s' % overlap[:10])
print('-' * 72)
if len(sub_ids) <= 5:
    print('sample_submission lists only the visible examples, so this cannot')
    print('settle the question. It stays open until a scored rerun.')
elif not overlap:
    print('NO OVERLAP. The contact-override exploit is unavailable to you here.')
    print('  Either the public scores rely on the rerun swapping in overlapping')
    print('  wells, or their gain comes from the modelling stack after all.')
else:
    print('OVERLAP PRESENT on %d wells. The exploit would fire.' % len(overlap))
    print('  Before using it, weigh the private-LB risk: if the private split does')
    print('  not share wells the way the public one does, anything built on this')
    print('  collapses, and it would be carrying your final submission.')
    print('  A split strategy is available: one final submission with it, one')
    print('  without. You get two.')
print('=' * 72)


## Reading G1

The decile table and the counterfactual sweep answer different questions. The
table describes what *is*; the sweep asks what the error would be if the test
distribution were shifted arbitrarily far toward isolation.

The number that matters is the upper bound. Your gap is 3.41 ft. If the most
extreme isolation shift buys only ~1 ft, then roughly 2.4 ft is caused by
something that has nothing to do with field support — and every hour spent
retuning the confidence taper is an hour spent on a third of the problem at most.

The three Spearman values at the end check whether `nn_dist` and `field_conf` are
measuring the same thing. If they correlate weakly with each other but both
correlate with error, they're separate axes and a two-variable taper has more room
than a one-variable one.

## What's left if G1 says "minor"

Candidates that don't depend on spatial support, roughly in order of how cheaply
they can be tested:

- **Bimodal wells.** Your ceiling probes keep finding ~0.5 ft of per-well
  heterogeneity that no observable identifies. A datum-scan with two plausible
  minima is a concrete, physically-motivated reason for that, and it's the one
  idea from the public notebooks you haven't got. Detection alone is testable
  without changing any prediction.
- **GR calibration drift.** Your own synthetic-corruption work reproduced the
  hard-subpopulation fingerprint by applying slow gain/offset drift along the
  blind zone. v22 fits the affine GR map once on the known zone and holds it
  fixed for the whole lateral. If the true gain drifts, error grows with distance
  from the heel — which is exactly the shape of a gap that local validation can't
  see, because local wells and test wells differ in how far the blind zone runs.
- **Blind-zone length.** Related and even cheaper: check whether per-well error
  correlates with blind-zone length in `v22_oof.pkl`. If it does strongly, and
  test wells have longer blind zones, that's a mechanical gap contributor you can
  measure today with data you already have.

## G3 — read their fitted models for the feature vocabulary

If `rogii-claude-models-pub` and `rogii-model-package` are public, mount them
(+ Add Input → Datasets → search the name) and run G3. It loads the boosters and
pulls `feature_name()` and `feature_importance('gain')`.

This is the one part of their work that transfers cleanly: their entire
engineered feature vocabulary, ranked by how much each actually earned — without
needing their code, and without depending on the layers we can't verify (the
contact override, the precomputed CSV).

**LightGBM text models carry `feature_names=` as plain text**, so the first
extraction path needs no library and can't fail on a version mismatch. CatBoost,
binary LightGBM, and pickled sklearn are tried in turn after that.

The output splits families into what v22 already has and what it lacks. The
second list is the whole point. My prior is that trajectory tortuosity appears
there — you have zero curvature features, and it was the single largest domain
feature in the independent toolkit we looked at earlier.

**If G3 finds nothing**, that is also an answer: either the datasets aren't
public, or they aren't mountable — and Section 2.6 requires external data be
equally accessible to all participants at no cost.


In [ ]:
# ===== G3: read the public notebooks' fitted models for their feature vocabulary =====
# Mount rogii-claude-models-pub and rogii-model-package (+ Add Input -> Datasets),
# then run. Read-only: loads models, extracts feature names + gain, and flags which
# families v22 lacks. Does not run or copy their code.
import os, glob, re
import numpy as np

EXTS = ('.txt', '.lgb', '.model', '.cbm', '.pkl', '.pickle', '.joblib', '.json', '.bin')
roots = [d for d in glob.glob('/kaggle/input/*') if os.path.isdir(d)]
print('mounted inputs:')
for d in roots:
    print('   ', os.path.basename(d))

cands = []
for d in roots:
    if 'wellbore-geology-prediction' in d or d.endswith('competitions'):
        continue
    for p in glob.glob(os.path.join(d, '**', '*'), recursive=True):
        if os.path.isfile(p) and p.lower().endswith(EXTS) and os.path.getsize(p) > 2048:
            cands.append(p)

print('\ncandidate model files: %d' % len(cands))
for p in cands[:25]:
    print('   %8.1f MB  %s' % (os.path.getsize(p) / 1e6, p.replace('/kaggle/input/', '')))
if not cands:
    print('\nNothing found. Either the datasets are not mounted, or they are private')
    print('(which would itself answer the question: Section 2.6 requires external')
    print('data to be equally accessible to all participants at no cost).')

FEATS = {}


def _add(names, gains, tag):
    if not names:
        return
    g = np.asarray(gains, float) if gains is not None else np.ones(len(names), float)
    if g.sum() > 0:
        g = 100.0 * g / g.sum()
    for n, gv in zip(names, g):
        FEATS[str(n)] = FEATS.get(str(n), 0.0) + float(gv)
    print('   + %-46s %3d features' % (tag, len(names)))


for p in cands:
    base = os.path.basename(p)
    # LightGBM text models carry feature_names= as plain text -> no library needed
    try:
        with open(p, 'r', errors='ignore') as fh:
            head = fh.read(600000)
        if 'feature_names=' in head:
            nm = re.search(r'feature_names=(.*)', head).group(1).split()
            _add(nm, None, base + ' [lgb text]')
            continue
    except Exception:
        pass
    try:
        import lightgbm as lgb
        b = lgb.Booster(model_file=p)
        _add(b.feature_name(), b.feature_importance('gain'), base + ' [lgb]')
        continue
    except Exception:
        pass
    try:
        from catboost import CatBoostRegressor
        m = CatBoostRegressor()
        m.load_model(p)
        _add(list(m.feature_names_), m.get_feature_importance(), base + ' [catboost]')
        continue
    except Exception:
        pass
    try:
        import pickle
        with open(p, 'rb') as fh:
            o = pickle.load(fh)
        for obj in (list(o.values()) if isinstance(o, dict) else [o]):
            nm = getattr(obj, 'feature_names_in_', None)
            if nm is None:
                nm = getattr(obj, 'feature_name_', None)
            if nm is not None:
                _add(list(nm), getattr(obj, 'feature_importances_', None),
                     base + ' [pickle]')
    except Exception:
        pass

if not FEATS:
    print('\nNo feature names recovered.')
else:
    order = sorted(FEATS.items(), key=lambda kv: -kv[1])
    print('\n%d distinct features recovered. Top 40 by gain:\n' % len(FEATS))
    for i, (n, gv) in enumerate(order[:40]):
        print('  %2d. %-52s %6.2f%%' % (i + 1, n[:52], gv))

    V22_HAS = {
        'gamma ray / GR':       ['gr', 'gamma'],
        'GR correlation (NCC)': ['ncc', 'corr', 'xcorr', 'match'],
        'depth / TVD':          [r'\bz\b', 'tvd', 'depth'],
        'measured depth':       ['md', 'measured'],
        'position X/Y':         [r'\bx\b', r'\by\b', 'east', 'north'],
        'heading / azimuth':    ['head', 'azim', 'hx', 'hy'],
        'dip / slope':          ['dip', 'slope', 'grad'],
        'structural field':     ['field', 'krig', 'idw', 'offset', 'neighb'],
        'isolation distance':   ['nn', 'dist'],
        'strat level u=T+Z':    ['u_', r'_u\b', 'strat', 'level'],
        'typewell':             ['tw', 'typewell', 'ref'],
    }
    V22_LACKS = {
        'TORTUOSITY / curvature': ['tort', 'curv', 'dogleg', 'dls', 'bend'],
        'inclination':            ['incl', 'build'],
        'GR despiking':           ['despike', 'spike', 'hampel', 'medfilt'],
        'distance correlation':   ['dcor', 'dist_corr'],
        'rolling stats':          ['roll', 'std', 'var', 'skew', 'kurt', 'mad'],
        'lag / difference':       ['lag', 'diff', 'delta', 'shift'],
        'entropy / complexity':   ['entrop', 'complex', 'fft', 'spect'],
    }

    def _hits(pats):
        tot, ex = 0.0, []
        for n, gv in FEATS.items():
            if any(re.search(p, n.lower()) for p in pats):
                tot += gv
                if len(ex) < 3:
                    ex.append(n[:30])
        return tot, ex

    print('\n' + '=' * 74)
    print('FEATURE FAMILIES v22 ALREADY HAS')
    print('-' * 74)
    for lab, pats in V22_HAS.items():
        t, ex = _hits(pats)
        if t > 0.05:
            print('  %-26s %6.2f%% gain   %s' % (lab, t, ', '.join(ex)))

    print('\nFEATURE FAMILIES v22 LACKS  <-- the actionable list')
    print('-' * 74)
    gaps = []
    for lab, pats in V22_LACKS.items():
        t, ex = _hits(pats)
        if t > 0.05:
            gaps.append((t, lab, ex))
    for t, lab, ex in sorted(gaps, reverse=True):
        print('  %-26s %6.2f%% gain   %s' % (lab, t, ', '.join(ex)))
    if not gaps:
        print('  none - their vocabulary is inside yours.')

    print('\n' + '=' * 74)
    if gaps and sorted(gaps, reverse=True)[0][0] > 5.0:
        top = sorted(gaps, reverse=True)[0]
        print('VERDICT: %s carries %.1f%% of total gain and v22 has' % (top[1], top[0]))
        print('  nothing in that family. That is the concrete, transferable gap.')
        print('  Build it as a confidence modulator on your existing rho /')
        print('  spatial_conf machinery rather than as a new branch.')
    elif gaps:
        print('VERDICT: gaps are all minor (<5% gain each). Their feature')
        print('  vocabulary is essentially yours. Nothing worth porting.')
    else:
        print('VERDICT: no feature families outside v22. Their edge is not features.')
    print('=' * 74)
